**Problem #1**

In [3]:
import numpy as np

# 1. Define the Augmented Matrix
# We use dtype=float to ensure division operations yield floating point numbers, not integers
A_aug = np.array([
    [ 4.0,  2.0, -1.0,  3.0,  1.0, 15.0],
    [ 2.0,  5.0,  2.0, -1.0,  3.0, 21.0],
    [-1.0,  2.0,  6.0,  2.0, -2.0,  9.0],
    [ 3.0, -1.0,  2.0,  7.0,  1.0, 19.0],
    [ 1.0,  3.0, -2.0,  1.0,  8.0, 17.0]
])

print("--- Initial Augmented Matrix ---")
print(A_aug)

n = len(A_aug)

# 2. Perform Row Operations to reach Row Echelon Form
for i in range(n):
    # Partial Pivoting: Find the maximum element in the current column to swap 
    # (Improves numerical stability)
    max_row = np.argmax(np.abs(A_aug[i:n, i])) + i
    if i != max_row:
        # Swap current row with the row containing the maximum element
        A_aug[[i, max_row]] = A_aug[[max_row, i]]
        
    # Make the pivot element 1 by dividing the whole row by the pivot
    pivot = A_aug[i, i]
    A_aug[i] = A_aug[i] / pivot
    
    # Eliminate the elements below the pivot
    for j in range(i + 1, n):
        factor = A_aug[j, i]
        A_aug[j] = A_aug[j] - factor * A_aug[i]

print("\n--- Matrix in Row Echelon Form ---")
np.set_printoptions(suppress=True, precision=4)
print(A_aug)

# 3. Back-Substitution to solve for unknowns
w = np.zeros(n)

# Loop backwards from the last row to the first
for i in range(n - 1, -1, -1):
    # The value is the constant at the end of the row minus the dot product 
    # of the known variables and their coefficients
    w[i] = A_aug[i, -1] - np.dot(A_aug[i, i+1:n], w[i+1:n])

print("\n--- Final Sensor Weighting Factors ---")
for i in range(n):
    print(f"w{i+1} = {w[i]:.4f}")

--- Initial Augmented Matrix ---
[[ 4.  2. -1.  3.  1. 15.]
 [ 2.  5.  2. -1.  3. 21.]
 [-1.  2.  6.  2. -2.  9.]
 [ 3. -1.  2.  7.  1. 19.]
 [ 1.  3. -2.  1.  8. 17.]]

--- Matrix in Row Echelon Form ---
[[ 1.      0.5    -0.25    0.75    0.25    3.75  ]
 [ 0.      1.      0.625  -0.625   0.625   3.375 ]
 [ 0.      0.      1.      0.7391  0.4203  3.7536]
 [ 0.      0.      0.      1.      1.7789  4.0476]
 [-0.     -0.     -0.     -0.      1.      2.2566]]

--- Final Sensor Weighting Factors ---
w1 = 3.7321
w2 = 0.2477
w3 = 2.7805
w4 = 0.0334
w5 = 2.2566


**Problem #2**

In [4]:
import numpy as np

# Dataset
distances = np.array([10, 20, 30, 40, 50, 60, 70, 80, 90, 100])
voltages = np.array([2.85, 2.42, 2.08, 1.82, 1.61, 1.44, 1.30, 1.18, 1.08, 0.99])

# --- 1. Direct Linear Interpolation Function ---
def direct_linear(x_target, x_data, y_data):
    # Find the nearest two points (indices)
    idx = np.where(x_data > x_target)[0][0]
    
    x_vals = x_data[idx-1:idx+1]
    y_vals = y_data[idx-1:idx+1]
    
    # Set up the matrix for a0 + a1*x = y
    # Matrix X: Column 1 is all 1s (for a0), Column 2 is x values (for a1)
    X = np.vstack([np.ones(2), x_vals]).T
    Y = y_vals
    
    # Solve for coefficients [a0, a1]
    coeffs = np.linalg.solve(X, Y)
    a0, a1 = coeffs
    
    # Apply polynomial
    y_target = a0 + a1 * x_target
    return y_target

# --- 2. Direct Quadratic Interpolation Function ---
def direct_quadratic(x_target, x_data, y_data):
    # Find the nearest three points
    closest_idx = np.abs(x_data - x_target).argmin()
    
    # Handle edges to ensure we grab 3 points
    if closest_idx == 0: idx = 1
    elif closest_idx == len(x_data) - 1: idx = len(x_data) - 2
    else: idx = closest_idx

    x_vals = x_data[idx-1:idx+2]
    y_vals = y_data[idx-1:idx+2]
    
    # Set up the matrix for a0 + a1*x + a2*x^2 = y
    # Matrix X: Col 1 is 1s, Col 2 is x, Col 3 is x^2
    X = np.vstack([np.ones(3), x_vals, x_vals**2]).T
    Y = y_vals
    
    # Solve for coefficients [a0, a1, a2]
    coeffs = np.linalg.solve(X, Y)
    a0, a1, a2 = coeffs
    
    # Apply polynomial
    y_target = a0 + a1 * x_target + a2 * (x_target**2)
    return y_target

# --- 3. Execute ---
targets = [55, 94.5]

print("--- Direct Linear Interpolation Results ---")
for t in targets:
    result = direct_linear(t, distances, voltages)
    print(f"Voltage at d = {t} cm: {result:.4f} V")

print("\n--- Direct Quadratic Interpolation Results ---")
for t in targets:
    result = direct_quadratic(t, distances, voltages)
    print(f"Voltage at d = {t} cm: {result:.5f} V")

--- Direct Linear Interpolation Results ---
Voltage at d = 55 cm: 1.5250 V
Voltage at d = 94.5 cm: 1.0395 V

--- Direct Quadratic Interpolation Results ---
Voltage at d = 55 cm: 1.52000 V
Voltage at d = 94.5 cm: 1.03826 V
